# Portfolio Risk & Return Analytics

**Разделы:**
1. Определение портфеля
2. Загрузка данных
3. Проверка качества данных
4. Доходности и базовая статистика
5. Волатильность и корреляционная матрица
6. VaR / CVaR
7. Sharpe / Sortino ratio
8. PCA на ковариационной матрице
9. Efficient Frontier (Markowitz)
10. Выводы


## 0. Импорты

In [1]:
import numpy as np
import pandas as pd
import yfinance as yf
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.optimize import minimize
from scipy import stats

plt.style.use('seaborn-v0_8-darkgrid')
pd.options.display.float_format = '{:.4f}'.format


## 1. Определение портфеля

Впиши сюда свои реальные тикеры и веса. Веса должны суммироваться в 1.0.


In [5]:
portfolio = {
    'PG': 0.15,
    'KHC': 0.10,
    'SIRI': 0.05,
    'OKTA': 0.15,
    'S': 0.30,
    'PFE': 0.20,
    'FUSD.L': 0.05,
    
}

assert abs(sum(portfolio.values()) - 1.0) < 1e-6, "Веса должны суммироваться в 1.0"

tickers = list(portfolio.keys())
weights = np.array(list(portfolio.values()))

print(f"Активов в портфеле: {len(tickers)}")
print(f"Сумма весов: {sum(portfolio.values()):.2f}")


Активов в портфеле: 7
Сумма весов: 1.00


## 2. Загрузка данных

Используем Adjusted Close (учитывает дивиденды и сплиты) минимум за 3-5 лет.


In [21]:
START_DATE = '2022-02-01'
END_DATE = '2026-07-25'

raw_data = yf.download(tickers, start=START_DATE, end=END_DATE, auto_adjust=True)
prices = raw_data['Close']


print(f"Период: {prices.index.min().date()} — {prices.index.max().date()}")
print(f"Количество торговых дней: {len(prices)}")
prices.head(1600)


[*********************100%***********************]  7 of 7 completed

Период: 2022-02-01 — 2026-07-24
Количество торговых дней: 1152


Ticker,FUSD.L,KHC,OKTA,PFE,PG,S,SIRI
Date,,,,,,,
2022-02-01,8.0926,28.8406,201.4200,41.4023,142.1181,46.8900,56.6393
2022-02-02,8.1628,29.0402,191.2400,42.0186,144.6627,43.8900,55.9690
2022-02-03,8.2058,28.6091,182.8000,41.6441,146.0327,40.9000,55.8852
2022-02-04,8.0405,27.7390,188.8600,41.3477,143.7107,42.8500,56.8069
2022-02-07,8.0846,27.7151,188.0900,41.5115,142.6342,43.6700,56.4717
...,...,...,...,...,...,...,...
2026-07-20,13.4500,25.8600,148.4100,24.3245,148.0250,19.4600,30.6000
2026-07-21,13.4500,25.7900,141.7100,24.5112,147.0026,18.8100,30.2900
2026-07-22,13.4750,25.9600,136.6900,24.3933,148.0250,18.2200,29.8300


## 3. Проверка качества данных

Обязательный шаг перед любым анализом — иначе метрики будут искажены.


In [22]:
# Пропущенные значения
missing = prices.isna().sum()
print("Пропущенные значения по тикерам:")
print(missing[missing > 0] if missing.sum() > 0 else "Пропусков нет")

# Заполнение небольших пропусков (праздники на разных биржах и т.п.)
prices_clean = prices.ffill().dropna()

# Проверка на аномальные скачки (потенциальные ошибки в данных)
daily_returns_check = prices_clean.pct_change().dropna()
extreme_moves = daily_returns_check[(daily_returns_check.abs() > 0.20)]
print(f"\nДней с движением >20% (проверить на ошибки данных): {extreme_moves.count().sum()}")
if extreme_moves.count().sum() > 0:
    display(extreme_moves.dropna(how='all'))


Пропущенные значения по тикерам:
Ticker
FUSD.L    23
KHC       30
OKTA      30
PFE       30
PG        30
S         30
SIRI      30
dtype: int64

Дней с движением >20% (проверить на ошибки данных): 7


Ticker,FUSD.L,KHC,OKTA,PFE,PG,S,SIRI
Date,,,,,,,
2022-09-01,NaN,NaN,-0.3370,NaN,NaN,NaN,NaN
2022-12-01,NaN,NaN,0.2646,NaN,NaN,NaN,NaN
2023-06-02,NaN,NaN,NaN,NaN,NaN,-0.3514,NaN
2023-07-20,NaN,NaN,NaN,NaN,NaN,NaN,0.4226
2024-02-29,NaN,NaN,0.2291,NaN,NaN,NaN,NaN
2025-03-04,NaN,NaN,0.2427,NaN,NaN,NaN,NaN
2026-05-29,NaN,NaN,0.3014,NaN,NaN,NaN,NaN


In [30]:
returns = prices.pct_change().dropna()


In [32]:
extreme_moves = returns[returns.abs().gt(0.2).any(axis=1)]
print(extreme_moves)

Ticker      FUSD.L     KHC    OKTA     PFE      PG       S    SIRI
Date                                                              
2022-09-01 -0.0127  0.0080 -0.3370  0.0310  0.0123 -0.0590  0.0066
2022-12-01  0.0245  0.0036  0.2646  0.0190  0.0006  0.0366 -0.0123
2023-06-02  0.0175  0.0084 -0.0115  0.0089  0.0178 -0.3514 -0.0324
2023-07-20  0.0011  0.0072 -0.0087  0.0091  0.0081 -0.0600  0.4226
2024-02-29  0.0031 -0.0073  0.2291 -0.0178 -0.0069 -0.0018 -0.0023
2025-03-04 -0.0269 -0.0109  0.2427 -0.0190 -0.0137 -0.0119 -0.0162
2026-05-29  0.0026 -0.0188  0.3014  0.0015 -0.0161 -0.0816 -0.0117


In [41]:
extreme_events = (
    returns.where(returns.abs() > 0.2)
    .melt(
        ignore_index=False,
        var_name="Ticker",
        value_name="Return"
    )
    .dropna()
    .reset_index()
)

extreme_events.columns = [
    "Date",
    "Ticker",
    "Return"
]

print(extreme_events)

extreme_events.to_csv(
    "../reports/extreme_events.csv",
    index=False)

        Date Ticker  Return
0 2022-09-01   OKTA -0.3370
1 2022-12-01   OKTA  0.2646
2 2024-02-29   OKTA  0.2291
3 2025-03-04   OKTA  0.2427
4 2026-05-29   OKTA  0.3014
5 2023-06-02      S -0.3514
6 2023-07-20   SIRI  0.4226


## 4. Доходности и базовая статистика

In [ ]:
returns = prices_clean.pct_change().dropna()

# Годовая доходность и волатильность по активам
annual_return = returns.mean() * 252
annual_vol = returns.std() * np.sqrt(252)

stats_table = pd.DataFrame({
    'Annual Return': annual_return,
    'Annual Volatility': annual_vol,
    'Sharpe (rf=0)': annual_return / annual_vol
}).sort_values('Sharpe (rf=0)', ascending=False)

stats_table


In [ ]:
# Доходность портфеля
portfolio_returns = (returns * weights).sum(axis=1)
cumulative_returns = (1 + portfolio_returns).cumprod()

fig, ax = plt.subplots(figsize=(12, 5))
cumulative_returns.plot(ax=ax, linewidth=2)
ax.set_title('Кумулятивная доходность портфеля')
ax.set_ylabel('Рост $1')
plt.tight_layout()
plt.show()


## 5. Волатильность и корреляционная матрица

In [ ]:
corr_matrix = returns.corr()

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm', center=0, ax=ax)
ax.set_title('Корреляционная матрица доходностей')
plt.tight_layout()
plt.show()


In [ ]:
# Ковариационная матрица (годовая) — нужна для VaR, PCA и Markowitz
cov_matrix_annual = returns.cov() * 252

portfolio_variance = weights @ cov_matrix_annual @ weights
portfolio_vol = np.sqrt(portfolio_variance)

print(f"Годовая волатильность портфеля: {portfolio_vol:.2%}")


## 6. Value at Risk (VaR) и Conditional VaR (CVaR)

Два метода: параметрический (предполагает нормальность) и исторический (без предположений о распределении).


In [ ]:
confidence_level = 0.95
portfolio_mean = portfolio_returns.mean()
portfolio_std = portfolio_returns.std()

# Параметрический VaR (предполагает нормальное распределение)
z_score = stats.norm.ppf(1 - confidence_level)
var_parametric = -(portfolio_mean + z_score * portfolio_std)

# Исторический VaR (эмпирический перцентиль, без предположений)
var_historical = -portfolio_returns.quantile(1 - confidence_level)

# CVaR (Expected Shortfall) — среднее по хвосту хуже VaR
cvar_historical = -portfolio_returns[portfolio_returns <= -var_historical].mean()

print(f"Daily VaR (95%, parametric): {var_parametric:.2%}")
print(f"Daily VaR (95%, historical): {var_historical:.2%}")
print(f"Daily CVaR (95%, historical): {cvar_historical:.2%}")
print(f"\nАннуализированный VaR (95%, historical): {var_historical * np.sqrt(252):.2%}")


## 7. Sharpe и Sortino Ratio

In [ ]:
risk_free_rate = 0.04  # подставь актуальную безрисковую ставку (например, 3-мес T-bill)

annual_portfolio_return = portfolio_returns.mean() * 252
annual_portfolio_vol = portfolio_returns.std() * np.sqrt(252)

sharpe_ratio = (annual_portfolio_return - risk_free_rate) / annual_portfolio_vol

# Sortino: учитывает только downside volatility, не наказывает за "хорошую" волатильность вверх
downside_returns = portfolio_returns[portfolio_returns < 0]
downside_vol = downside_returns.std() * np.sqrt(252)
sortino_ratio = (annual_portfolio_return - risk_free_rate) / downside_vol

print(f"Годовая доходность портфеля: {annual_portfolio_return:.2%}")
print(f"Годовая волатильность портфеля: {annual_portfolio_vol:.2%}")
print(f"Sharpe Ratio: {sharpe_ratio:.2f}")
print(f"Sortino Ratio: {sortino_ratio:.2f}")


## 8. PCA на ковариационной матрице

Показывает, сколько "независимых источников риска" реально определяют движение портфеля —
если первая компонента объясняет 70%+ дисперсии, портфель менее диверсифицирован, чем кажется по числу активов.


In [ ]:
from sklearn.decomposition import PCA

pca = PCA()
pca.fit(returns)

explained_var = pca.explained_variance_ratio_
cumulative_var = np.cumsum(explained_var)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].bar(range(1, len(explained_var) + 1), explained_var)
axes[0].set_title('Explained Variance по компонентам')
axes[0].set_xlabel('Главная компонента')
axes[0].set_ylabel('Доля объяснённой дисперсии')

axes[1].plot(range(1, len(cumulative_var) + 1), cumulative_var, marker='o')
axes[1].axhline(0.8, color='red', linestyle='--', label='80% порог')
axes[1].set_title('Кумулятивная объяснённая дисперсия')
axes[1].set_xlabel('Число компонент')
axes[1].legend()

plt.tight_layout()
plt.show()

print(f"Первая компонента объясняет {explained_var[0]:.1%} дисперсии портфеля")


## 9. Efficient Frontier (Марковиц)

Строим границу эффективных портфелей методом Monte Carlo + точную оптимизацию через scipy.


In [ ]:
n_assets = len(tickers)
n_portfolios = 5000

results = np.zeros((3, n_portfolios))
all_weights = np.zeros((n_portfolios, n_assets))

np.random.seed(42)
for i in range(n_portfolios):
    w = np.random.random(n_assets)
    w /= w.sum()
    all_weights[i, :] = w

    port_return = np.sum(w * annual_return)
    port_vol = np.sqrt(w @ cov_matrix_annual @ w)

    results[0, i] = port_return
    results[1, i] = port_vol
    results[2, i] = (port_return - risk_free_rate) / port_vol  # Sharpe

fig, ax = plt.subplots(figsize=(12, 7))
scatter = ax.scatter(results[1], results[0], c=results[2], cmap='viridis', alpha=0.6, s=10)
plt.colorbar(scatter, label='Sharpe Ratio')

# Твой текущий портфель на графике
ax.scatter(portfolio_vol, np.sum(weights * annual_return), color='red', marker='*', s=400,
           label='Мой текущий портфель', edgecolors='black', linewidths=1)

ax.set_xlabel('Волатильность (годовая)')
ax.set_ylabel('Ожидаемая доходность (годовая)')
ax.set_title('Efficient Frontier: 5000 случайных портфелей')
ax.legend()
plt.tight_layout()
plt.show()


In [ ]:
# Точная оптимизация: портфель с максимальным Sharpe ratio
def neg_sharpe(w, mean_returns, cov_matrix, rf):
    port_return = np.sum(w * mean_returns)
    port_vol = np.sqrt(w @ cov_matrix @ w)
    return -(port_return - rf) / port_vol

constraints = ({'type': 'eq', 'fun': lambda w: np.sum(w) - 1})
bounds = tuple((0, 1) for _ in range(n_assets))
init_guess = np.array(n_assets * [1. / n_assets])

optimal = minimize(neg_sharpe, init_guess, args=(annual_return, cov_matrix_annual, risk_free_rate),
                    method='SLSQP', bounds=bounds, constraints=constraints)

optimal_weights = pd.Series(optimal.x, index=tickers).sort_values(ascending=False)
print("Оптимальные веса (максимальный Sharpe ratio):")
print(optimal_weights)


## 10. Выводы

*(Заполни после прогона на своих реальных данных)*

- Сравни свой текущий портфель с оптимальным по Sharpe — насколько ты близок к efficient frontier?
- Какая доля риска объясняется первой главной компонентой (концентрация риска)?
- Какие активы дают наибольший вклад в диверсификацию (низкая корреляция + позитивный Sharpe)?
- Ограничения анализа: исторические данные не гарантируют будущих корреляций и доходностей;
  здесь не учтены транзакционные издержки при ребалансировке.
